# 01 - Extract Pushed ZIPs to Date-Partitioned CSVs & 7-Day Cleanup (Task 1)
Configured as **Task 1 (`extract_to_landing_and_cleanup`)** in your Multi-Task Databricks Workflow Job.

### Why We Extract Directly to `.csv` Here:
1. **Zero Duplicate `.zip` Storage:** Takes `.zip` files from `/Volumes/workspace/bronze/landing/_manual_upload/` (or any existing `.zip` in `Dispatch_SCADA/`), extracts the inner CSV into `/Volumes/workspace/bronze/landing/Dispatch_SCADA/YYYY-MM-DD/<file_stem>.csv`, and **deletes the `.zip` immediately**.
2. **Strips AEMO's Row-1 `C` Header:** Removes line 1 (`C,"NEMP.WORLD"...`) during extraction so that line 1 of the saved `.csv` is the true column header (`I,DISPATCH,UNIT_SCADA,1,SETTLEMENTDATE,DUID,SCADAVALUE,...`). This makes the `.csv` files **100% compatible with native Databricks Delta Live Tables (DLT) Auto Loader (`cloudFiles`)**!
3. **Updates `workspace.bronze.meta_file_control_log`** and **purges extracted `.csv` files older than 7 days**.

### Cell 1: Imports & Unity Catalog Configuration

In [0]:
import json
import re
import shutil
import zipfile
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

In [0]:
# AEMO NEM operates strictly on AEST (UTC+10) year-round (no Daylight Saving)
NEM_TZ = timezone(timedelta(hours=10))

CATALOG = "aemo"
SCHEMA = "bronze"
retention_days = 7
VOLUME_NAME = "landing"
LANDING_BASE_DIR = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}")
MANUAL_UPLOAD_DIR = LANDING_BASE_DIR / "_manual_upload"
SCADA_LANDING_DIR = LANDING_BASE_DIR / "Dispatch_SCADA"
CONTROL_LOG_TABLE = f"{CATALOG}.{SCHEMA}.meta_file_control_log"

# Ensure Unity Catalog Schema, Managed Volume, and Control Table exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONTROL_LOG_TABLE} (
        file_name STRING,
        feed_name STRING,
        landing_date_folder STRING,
        landing_path STRING,
        file_size_bytes BIGINT,
        status STRING,
        downloaded_at_utc STRING,
        ingested_at_utc STRING,
        purged_at_utc STRING,
        bronze_row_count BIGINT
    )
    USING DELTA
""")
MANUAL_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
SCADA_LANDING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Fabric Push Dropzone : {MANUAL_UPLOAD_DIR}")
print(f"Extracted CSV Folder : {SCADA_LANDING_DIR}")
print(f"Control Log Table    : {CONTROL_LOG_TABLE}")

In [0]:
# %%sql
# truncate table aemo.bronze.meta_file_control_log

In [0]:
# # Delete all files and subfolders in a Unity Catalog volume
# from pathlib import Path
# import shutil

# # Volume root (update as needed)
# CATALOG = "aemo"
# SCHEMA = "bronze"
# VOLUME_NAME = "landing/Dispatch_SCADA"
# VOLUME_ROOT = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}")

# for item in VOLUME_ROOT.iterdir():
#     if item.is_dir():
#         shutil.rmtree(item, ignore_errors=True)
#     else:
#         item.unlink(missing_ok=True)
# print(f"All contents of {VOLUME_ROOT} have been deleted.")

### Cell 2: Check `workspace.bronze.meta_file_control_log` for Previously Processed Files

In [0]:
df_existing_log = spark.table(CONTROL_LOG_TABLE).toPandas()
known_files = set(df_existing_log["file_name"].dropna().unique())
print(f"Found {len(known_files):,} tracked files in {CONTROL_LOG_TABLE}.")
if not df_existing_log.empty:
    display(df_existing_log.sort_values("downloaded_at_utc", ascending=False).head(10))

### Cell 3: Unzip `.zip` Files Directly into Clean Date-Partitioned `.csv` Files & Delete `.zip` Copies
Extracts the inner CSV, removes the first `C,"NEMP.WORLD"...` line so the `I,DISPATCH,UNIT_SCADA,...` header is on line 1, saves to `/Volumes/workspace/bronze/landing/Dispatch_SCADA/YYYY-MM-DD/<file_stem>.csv`, and deletes the `.zip` archive.

In [0]:
def extract_settlement_date_str(filename: str, fallback_dt: datetime) -> str:
    match = re.search(r"_(\d{4})(\d{2})(\d{2})\d{4}_", filename)
    if match:
        return f"{match.group(1)}-{match.group(2)}-{match.group(3)}"
    match_archive = re.search(r"#(\d{4})(\d{2})(\d{2})\d{4}", filename)
    if match_archive:
        return f"{match_archive.group(1)}-{match_archive.group(2)}-{match_archive.group(3)}"
    return fallback_dt.strftime("%Y-%m-%d")

def extract_aemo_zip_to_clean_csv(zip_path: Path, target_csv_path: Path) -> tuple[int, int]:
    with zipfile.ZipFile(zip_path, "r") as zf:
        csv_members = [m for m in zf.namelist() if m.lower().endswith(".csv")]
        if not csv_members:
            raise ValueError(f"No .CSV member found inside {zip_path}")
        raw_bytes = zf.read(csv_members[0])

    text = raw_bytes.decode("utf-8", errors="replace")
    lines = text.splitlines()

    # Strip the leading 'C,' metadata line so line 1 is the 'I,' header row
    header_idx = 0
    for idx, line in enumerate(lines):
        if line.startswith("I,"):
            header_idx = idx
            break

    clean_lines = lines[header_idx:]
    d_row_count = sum(1 for line in clean_lines[1:] if line.startswith("D,"))

    clean_bytes = ("\n".join(clean_lines) + "\n").encode("utf-8")
    target_csv_path.parent.mkdir(parents=True, exist_ok=True)
    target_csv_path.write_bytes(clean_bytes)

    return len(clean_bytes), d_row_count

now_utc = datetime.now(timezone.utc)
now_nem = datetime.now(NEM_TZ)
now_iso = now_utc.isoformat()

new_log_records = []
updated_log_records = []
extracted_count = 0
skipped_count = 0

# 1. Extract new .zip files from `_manual_upload` (pushed from Fabric)
pushed_zips = sorted(MANUAL_UPLOAD_DIR.glob("*.zip"))
for src_zip in pushed_zips:
    date_folder = extract_settlement_date_str(src_zip.name, now_nem)
    target_csv = SCADA_LANDING_DIR / date_folder / f"{src_zip.stem}.csv"

    if src_zip.name in known_files and target_csv.exists():
        skipped_count += 1
        src_zip.unlink(missing_ok=True)
        continue
    
    try:
        csv_size, d_rows = extract_aemo_zip_to_clean_csv(src_zip, target_csv)
    except Exception as e:
        print(src_zip, '-', e, '-', now_iso)
        err_df = spark.createDataFrame([(str(src_zip), str(e), now_iso)], schema=["zip_file", "error", "timestamp"])
        err_df.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.unzip_erros")
    src_zip.unlink(missing_ok=True)  # Delete the .zip immediately!
    extracted_count += 1
    print(f"Extracted {src_zip.name} -> {target_csv} ({d_rows:,} 'D' rows) & deleted ZIP")

    rec = {
        "file_name": src_zip.name,
        "feed_name": "Dispatch_SCADA",
        "landing_date_folder": date_folder,
        "landing_path": str(target_csv),
        "file_size_bytes": int(csv_size),
        "status": "EXTRACTED_TO_LANDING",
        "downloaded_at_utc": now_iso,
        "ingested_at_utc": now_iso,
        "purged_at_utc": None,
        "bronze_row_count": int(d_rows),
    }
    if src_zip.name in known_files:
        updated_log_records.append(rec)
    else:
        new_log_records.append(rec)
        known_files.add(src_zip.name)

# 2. Also convert any existing .zip files already sitting inside `Dispatch_SCADA/YYYY-MM-DD/`
for existing_zip in sorted(SCADA_LANDING_DIR.rglob("*.zip")):
    date_folder = extract_settlement_date_str(existing_zip.name, now_nem)
    target_csv = existing_zip.with_suffix(".csv")

    try:
        csv_size, d_rows = extract_aemo_zip_to_clean_csv(existing_zip, target_csv)
    except Exception as e:
        print(src_zip, '-', e, '-', now_iso)
        err_df = spark.createDataFrame([(str(src_zip), str(e), now_iso)], schema=["zip_file", "error", "timestamp"])
        err_df.write.format("delta").mode("append").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.unzip_erros")
    existing_zip.unlink(missing_ok=True)  # Remove duplicate .zip
    extracted_count += 1
    print(f"Converted existing ZIP {existing_zip.name} -> {target_csv.name} ({d_rows:,} 'D' rows) & deleted ZIP")

    rec = {
        "file_name": existing_zip.name,
        "feed_name": "Dispatch_SCADA",
        "landing_date_folder": date_folder,
        "landing_path": str(target_csv),
        "file_size_bytes": int(csv_size),
        "status": "EXTRACTED_TO_LANDING",
        "downloaded_at_utc": now_iso,
        "ingested_at_utc": now_iso,
        "purged_at_utc": None,
        "bronze_row_count": int(d_rows),
    }
    if existing_zip.name in known_files:
        updated_log_records.append(rec)
    else:
        new_log_records.append(rec)
        known_files.add(existing_zip.name)

print(f"\nTotal CSVs extracted: {extracted_count} | Skipped: {skipped_count}")

### Cell 4: Update `workspace.bronze.meta_file_control_log` & Run 7-Day CSV Cleanup

In [0]:
CONTROL_LOG_SCHEMA = T.StructType([
    T.StructField("file_name", T.StringType(), False),
    T.StructField("feed_name", T.StringType(), True),
    T.StructField("landing_date_folder", T.StringType(), True),
    T.StructField("landing_path", T.StringType(), True),
    T.StructField("file_size_bytes", T.LongType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("downloaded_at_utc", T.StringType(), True),
    T.StructField("ingested_at_utc", T.StringType(), True),
    T.StructField("purged_at_utc", T.StringType(), True),
    T.StructField("bronze_row_count", T.LongType(), True),
])

# log file update
# UPSERT START
if new_log_records:
    sdf_new = spark.createDataFrame(pd.DataFrame(new_log_records), schema=CONTROL_LOG_SCHEMA)
    sdf_new.write.format("delta").mode("append").saveAsTable(CONTROL_LOG_TABLE)

if updated_log_records:
    sdf_upd = spark.createDataFrame(pd.DataFrame(updated_log_records), schema=CONTROL_LOG_SCHEMA)
    dt_log = DeltaTable.forName(spark, CONTROL_LOG_TABLE)
    (
        dt_log.alias("target")
        .merge(sdf_upd.alias("source"), "target.file_name = source.file_name")
        .whenMatchedUpdate(set={
            "landing_path": "source.landing_path",
            "file_size_bytes": "source.file_size_bytes",
            "status": "source.status",
            "ingested_at_utc": "source.ingested_at_utc",
            "bronze_row_count": "source.bronze_row_count",
        })
        .execute()
    )
# UPSERT END

# 7-Day Retention Cleanup on Extracted CSVs
cutoff_date_str = (now_utc.astimezone(NEM_TZ) - timedelta(days=int(retention_days))).strftime("%Y-%m-%d")
eligible_df = (
    spark.table(CONTROL_LOG_TABLE)
    .filter(f"status IN ('EXTRACTED_TO_LANDING', 'INGESTED_TO_BRONZE') AND landing_date_folder < '{cutoff_date_str}'")
    .toPandas()
)

purged_updates = []
folders_removed = 0
for _, row in eligible_df.iterrows():
    fp = Path(row["landing_path"])
    if fp.exists():
        fp.unlink()
    purged_updates.append({
        "file_name": str(row["file_name"]),
        "status": "PURGED_FROM_LANDING",
        "purged_at_utc": now_iso,
    })

if SCADA_LANDING_DIR.exists():
    for date_dir in SCADA_LANDING_DIR.iterdir():
        if date_dir.is_dir() and date_dir.name < cutoff_date_str and not any(date_dir.iterdir()):
            shutil.rmtree(date_dir, ignore_errors=True)
            folders_removed += 1

if purged_updates:
    sdf_purged = spark.createDataFrame(pd.DataFrame(purged_updates))
    dt_log = DeltaTable.forName(spark, CONTROL_LOG_TABLE)
    (
        dt_log.alias("target")
        .merge(sdf_purged.alias("source"), "target.file_name = source.file_name")
        .whenMatchedUpdate(set={"status": "source.status", "purged_at_utc": "source.purged_at_utc"})
        .execute()
    )

display(spark.table(CONTROL_LOG_TABLE).orderBy(col("downloaded_at_utc").desc()).limit(15))

result = {
    "status": "OK",
    "extracted_csv_files": extracted_count,
    "skipped_already_processed": skipped_count,
    "purged_files": len(purged_updates),
    "timestamp_utc": now_iso,
}
try:
    dbutils.jobs.taskValues.set(key="step_1_summary", value=result)
    dbutils.notebook.exit(json.dumps(result))
except Exception:
    pass